# 15-тарау. Мәтін және тілдік модельдер

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 15.1. Тілдік модель деген не?

## 15.2. Мәтінді сандарға айналдыру: токендер

In [ ]:
corpus = """
бүгін ауа райы жылы . бүгін ауа райы суық . ертең ауа райы жылы болады .
бүгін күн ашық . ертең күн бұлтты болады . мен мектепке барамын .
мен кітап оқимын . мен математиканы жақсы көремін .
мен информатиканы жақсы көремін .
біз мектепке барамыз . біз кітап оқимыз . сен кітап оқисың ба .
мысық сүт ішеді . ит сүйек жейді . мысық ұйықтап жатыр . ит үреді .
бүгін біз жасанды интеллектті үйренеміз .
ертең біз нейрондық желіні жазамыз .
""".split()

vocab = sorted(set(corpus))                     # бірегей сөздер
word2id = {w: i for i, w in enumerate(vocab)}   # сөз → нөмір

print("Барлық сөз:", len(corpus), " Сөздік көлемі:", len(vocab))
sentence = "мен кітап оқимын"
print([word2id[w] for w in sentence.split()])

## 15.3. Ең қарапайым тілдік модель

In [ ]:
from collections import Counter, defaultdict

next_words = defaultdict(Counter)
for a, b in zip(corpus[:-1], corpus[1:]):     # көршілес сөз жұптары
    next_words[a][b] += 1

def next_probs(word):
    counts = next_words[word]
    total = sum(counts.values())
    return {w: n / total for w, n in counts.most_common()}

for w in ["бүгін", "ауа", "мен"]:
    probs = next_probs(w)
    print(f"{w} →", ", ".join(f"{k} {p:.2f}" for k, p in probs.items()))

In [ ]:
import numpy as np
rng = np.random.default_rng(7)

def generate(start, max_words=8):
    words = [start]
    for _ in range(max_words):
        probs = next_probs(words[-1])
        if not probs:
            break
        choice = rng.choice(list(probs.keys()), p=list(probs.values()))
        words.append(choice)
        if choice == ".":
            break
    return " ".join(words)

for start in ["бүгін", "мен", "ертең", "мысық"]:
    print(generate(start))

## 15.4. Эмбеддингтер: сөздің мағынасы — вектор

In [ ]:
emb = {
    "мысық":  np.array([0.9, 0.1, 0.2]),
    "ит":     np.array([0.9, 0.1, 0.4]),
    "жылқы":  np.array([0.8, 0.0, 0.9]),
    "алма":   np.array([0.0, 0.9, 0.1]),
    "алмұрт": np.array([0.0, 0.9, 0.2]),
}

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

pairs = [("мысық", "ит"), ("мысық", "жылқы"),
         ("мысық", "алма"), ("алма", "алмұрт")]
for a, b in pairs:
    print(f"{a:6s} ~ {b:6s}: {cosine(emb[a], emb[b]):.2f}")

## 15.5. Назар аудару механизмі

In [ ]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def attention(Q, K, V):
    d = Q.shape[-1]
    weights = softmax(Q @ K.T / np.sqrt(d))   # кім кімге назар аударады
    return weights @ V, weights

# 3 токен, әрқайсысы 2 өлшемді вектор
Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = Q.copy()
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])

output, weights = attention(Q, K, V)
print("Назар салмақтары:\n", weights.round(2))
print("Жаңа көріністер:\n", output.round(2))

## 15.6. Трансформер

## 15.7. Үлкен тілдік модельдер қалай оқытылады?